# ch04 卷积与滤波（练习版）

> 按提示补全 `____`，跑通所有 assert。`gray`（(384,512) uint8）与 `f32`（float32 版）
> 已在 setup 里备好，`correlate_zero_pad` / `max_abs_diff` / `gap_to_clean` 脚手架可用。
>
> 注意：挖空块内每条语句保持**单行**；循环 / 分支写在挖空块之外（见任务 4、8）。
> 题目真值都来自实跑（opencv 5.0.0），照提示写就能对上。

In [ ]:
from pathlib import Path

import warnings

import numpy as np

warnings.simplefilter("ignore")

import cv2

DATA = Path("data")
CL = DATA / "classic"

In [ ]:
home = cv2.imread(str(CL / "home.jpg"))
gray = cv2.cvtColor(home, cv2.COLOR_BGR2GRAY)   # (384, 512) uint8，滤波素材
f32 = gray.astype(np.float32)                   # 线性滤波统一在 float32 上做
print("home", home.shape, "| 灰度", gray.shape, gray.dtype,
      "| 均值", round(float(gray.mean()), 4), "| std", round(float(gray.std()), 4))

In [ ]:
# 脚手架：写在 @@todo 之外，练习版原样保留
def correlate_zero_pad(img, k):
    """手写 2D 相关（零填充 + 逐位置点积），与 cv2.filter2D 同口径。

    注意：这是「相关」（correlation）——核不翻转。要算真正的卷积，把 k 翻转再传进来。
    """
    kh, kw = k.shape
    ph, pw = kh // 2, kw // 2
    pad = np.pad(np.asarray(img, np.float64), ((ph, ph), (pw, pw)), mode="constant")
    h, w = img.shape
    out = np.zeros((h, w), np.float64)
    for i in range(h):
        for j in range(w):
            out[i, j] = float((pad[i:i + kh, j:j + kw] * k).sum())
    return out


def max_abs_diff(a, b):
    return round(float(np.abs(np.asarray(a, np.float64) - np.asarray(b, np.float64)).max()), 6)


def gap_to_clean(a, clean):
    """与干净图的平均绝对差（去噪效果的第二把尺子）。"""
    return round(float(np.abs(np.asarray(a, np.int64) - np.asarray(clean, np.int64)).mean()), 4)


print("脚手架就绪：correlate_zero_pad / max_abs_diff / gap_to_clean")

## 任务 1：手写相关 vs `filter2D`

再证明 `filter2D` 是相关不是卷积。

In [ ]:
ks = np.ones((3, 3), np.float32) / 9.0
ka = np.array([[1, 2], [3, 4]], np.float32)
tiny = gray[:40, :40]
cv_sym = cv2.filter2D(tiny.astype(np.float32), -1, ks, borderType=cv2.BORDER_CONSTANT)
cv_asym = cv2.filter2D(tiny.astype(np.float32), -1, ka, borderType=cv2.BORDER_CONSTANT)
d_sym = max_abs_diff(correlate_zero_pad(tiny, ks), cv_sym)
d_asym_corr = max_abs_diff(correlate_zero_pad(tiny, ka), cv_asym)
d_asym_conv = max_abs_diff(correlate_zero_pad(tiny, ka[::-1, ::-1]), cv_asym)
print("对称核   手写相关 vs filter2D 最大差:", d_sym)
print("非对称核 手写相关 vs filter2D 最大差:", d_asym_corr)
print("非对称核 手写卷积(翻核) vs filter2D 最大差:", d_asym_conv)
print("解读: 前两行几乎为 0（浮点误差），第三行差出 352 —— 说明 cv2.filter2D 做的是「相关」"
      "（核不翻转）而不是数学意义的卷积。对称核看不出区别，一旦用非对称核"
      "（如 Sobel、Gabor、方向性核）就必须想清楚自己是哪个口径")

assert d_sym <= 1e-4 and d_asym_corr == 0.0
assert d_asym_conv == 352.0

## 任务 2：核的 sum 决定亮度

`blur` 对账 + 未归一化饱和。

In [ ]:
k5 = np.ones((5, 5), np.float32) / 25.0
box_norm = cv2.blur(gray, (5, 5))
box_unnorm = cv2.boxFilter(gray, -1, (5, 5), normalize=False)
f2d_blur = cv2.filter2D(gray, -1, k5)
kk = np.array([[1, 2, 1], [2, 4, 2], [1, 2, 1]], np.float32)
ksum = float(kk.sum())
ksum_norm = round(float((kk / kk.sum()).sum()), 6)
m_norm = round(float(box_norm.mean()), 4)
m_unnorm = round(float(box_unnorm.mean()), 4)
d_blur_f2d = int(np.abs(box_norm.astype(int) - f2d_blur.astype(int)).max())
print("原图均值:", round(float(gray.mean()), 4))
print("blur(5,5) 均值:", m_norm, "| filter2D(均值核) 均值:", round(float(f2d_blur.mean()), 4),
      "| 两者最大差:", d_blur_f2d)
print("boxFilter 未归一化 均值:", m_unnorm, "| 唯一值数:", len(np.unique(box_unnorm)))
print("3x3 高斯样核 sum:", ksum, "→ 归一化后 sum:", ksum_norm)
print("解读: blur == filter2D(均值核)，逐像素最大差 0，两套写法完全等价。"
      "核的所有元素之和就是「直流增益」：sum=1 亮度不变；boxFilter 不归一化时 sum=25，"
      "结果被放大 25 倍直接饱和到 255（唯一值只剩 64 个）——这是新手最常踩的坑")

assert d_blur_f2d == 0 and m_norm == 116.1955
assert m_unnorm == 254.9622 and ksum == 16.0 and ksum_norm == 1.0

## 任务 3：高斯核与可分离性

sigma 影响 + 外积 + `sepFilter2D`。

In [ ]:
k1 = cv2.getGaussianKernel(5, 1.0)
k2 = cv2.getGaussianKernel(5, 2.0)
k5s = cv2.getGaussianKernel(5, 5.0)
k0 = cv2.getGaussianKernel(5, 0)
g2d = k1 @ k1.T
sum1, sum2 = round(float(k1.sum()), 6), round(float(g2d.sum()), 6)
d_sep = max_abs_diff(cv2.filter2D(f32, -1, g2d), cv2.sepFilter2D(f32, -1, k1, k1))
first1 = round(float(k1.ravel()[0]), 6)
first5 = round(float(k5s.ravel()[0]), 6)
auto0 = [round(float(v), 6) for v in k0.ravel()]
print("sigma=1 的 1D 核:", [round(float(v), 6) for v in k1.ravel()], "sum", sum1)
print("sigma=2 的 1D 核:", [round(float(v), 6) for v in k2.ravel()])
print("sigma=5 的 1D 核:", [round(float(v), 6) for v in k5s.ravel()])
print("外积核 sum:", sum2, "| 2D 滤波 vs sepFilter2D 最大差:", d_sep)
print("sigma=0（自动推算，即二项式系数）:", auto0)
print("解读: sigma 越大核越「平」（sigma=1 中心权重 0.4026，sigma=5 只有 0.2080）——"
      "平滑更强但更糊。1D 高斯的外积恰好等于 2D 高斯核，所以可以拆成横竖两趟做，"
      "这正是 sepFilter2D 快 15.5 倍的原理。sigma=0 时 OpenCV 按 ksize 反推，"
      "ksize=5 给出的是标准二项式系数 [0.0625, 0.25, 0.375, 0.25, 0.0625]")

assert sum1 == 1.0 and sum2 == 1.0 and d_sep <= 1e-4
assert first1 == 0.054489 and first5 == 0.192051
assert auto0 == [0.0625, 0.25, 0.375, 0.25, 0.0625]

## 任务 4：边界模式

循环骨架已给好，块内写 3 行。

In [ ]:
k3e = np.ones((3, 3), np.float32) / 9.0
k5e = np.ones((5, 5), np.float32) / 25.0
base5 = cv2.filter2D(f32, -1, k5e, borderType=cv2.BORDER_REFLECT_101)
modes = {"REFLECT": cv2.BORDER_REFLECT, "REPLICATE": cv2.BORDER_REPLICATE,
         "CONSTANT": cv2.BORDER_CONSTANT}
border_stats = {}
for nm, bt in modes.items():
    out = cv2.filter2D(f32, -1, k5e, borderType=bt)
    d = np.abs(out - base5)
    border_stats[nm] = (round(float(d.max()), 4), int((d > 1e-6).sum()), bool((d[2:-2, 2:-2] < 1e-6).all()))
# 半径 1 时 REFLECT 与 REPLICATE 是否等价
r1_reflect = cv2.filter2D(f32, -1, k3e, borderType=cv2.BORDER_REFLECT)
r1_replicate = cv2.filter2D(f32, -1, k3e, borderType=cv2.BORDER_REPLICATE)
r1_same = int((np.abs(r1_reflect - r1_replicate) > 1e-6).sum())
# copyMakeBorder 直观展示（含 filter2D 不支持的 WRAP）
bor = {nm: cv2.copyMakeBorder(gray, 2, 2, 2, 2, bt)[:2, :4] for nm, bt in
       [("REFLECT_101", cv2.BORDER_REFLECT_101), ("REFLECT", cv2.BORDER_REFLECT),
        ("REPLICATE", cv2.BORDER_REPLICATE), ("CONSTANT", cv2.BORDER_CONSTANT),
        ("WRAP", cv2.BORDER_WRAP)]}
print("原图左上 2x4:", gray[:2, :4].tolist())
for nm, im in bor.items():
    print(f"   copyMakeBorder {nm:12s} 左上 2x4:", im.tolist())
for nm, (mx, cnt, inner) in border_stats.items():
    print(f"   {nm:10s} vs 默认: 最大差 {mx} | 差异像素 {cnt} | 内圈完全一致 {inner}")
print("半径 1 时 REFLECT vs REPLICATE 差异像素:", r1_same)
print("解读: 边界模式只影响最外 k 圈（内圈逐像素一致），但对边缘目标影响很大——"
      "CONSTANT 把边缘当 0，最大差能到 112.2。反直觉的是半径 1 时 REFLECT 与 REPLICATE "
      "完全等价（差异 0）：REFLECT 会把边缘像素本身也镜像一次，一步之内就是复制。"
      "另外 BORDER_WRAP / TRANSPARENT 在 filter2D 里不支持（会抛异常），"
      "要手工用 copyMakeBorder 铺边")

assert border_stats["REPLICATE"] == (16.88, 2143, True)
assert border_stats["CONSTANT"] == (112.2, 3568, True)
assert r1_same == 0

## 任务 5：两把尺子评估去噪

`std` vs `gap_to_clean`。

In [ ]:
rs = np.random.RandomState(3)
m = rs.rand(*gray.shape)
sp = gray.copy()
sp[m < 0.02] = 0
sp[m > 0.98] = 255
gn = np.clip(gray + rs.normal(0, 20, gray.shape), 0, 255).astype(np.uint8)
filters = lambda src: {"中值5": cv2.medianBlur(src, 5),
                       "均值5": cv2.blur(src, (5, 5)),
                       "双边": cv2.bilateralFilter(src, 5, 50, 50)}
sp_gap = {k: gap_to_clean(v, gray) for k, v in filters(sp).items()}
gn_gap = {k: gap_to_clean(v, gray) for k, v in filters(gn).items()}
sp_std = {k: round(float(v.std()), 4) for k, v in filters(sp).items()}
gn_std = {k: round(float(v.std()), 4) for k, v in filters(gn).items()}
print("椒盐 2%: gap 原", gap_to_clean(sp, gray), "|", sp_gap, "| std 原", round(float(sp.std()), 4), sp_std)
print("高斯 σ=20: gap 原", gap_to_clean(gn, gray), "|", gn_gap, "| std 原", round(float(gn.std()), 4), gn_std)
print("解读: std 是「图有多平」，gap 是「离真值多远」，两把尺子会打架——"
      "椒盐图上均值滤波的 std 最低（41.1108 < 中值 43.7865），但 gap 反而最大（9.2347 > 5.8994），"
      "因为它把 0/255 抹成中间灰，看着平其实是糊。"
      "更反直觉的是：这个噪声密度下，「不做任何滤波」的 gap 只有 5.0841，比中值滤波还小——"
      "原图只被污染 2%，滤波却把整幅图都模糊了一遍，去噪收益抵不过模糊代价")

assert sp_gap == {"中值5": 5.8994, "均值5": 9.2347, "双边": 8.02}
assert gn_gap == {"中值5": 8.7138, "均值5": 8.5429, "双边": 7.5279}
assert sp_std["均值5"] == 41.1108 and sp_std["中值5"] == 43.7865
assert gap_to_clean(sp, gray) == 5.0841

## 任务 6：锐化核与拉普拉斯核

看 sum 判断亮度行为。

In [ ]:
sharp_k = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], np.float32)
lapl_k = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], np.float32)
sharped = cv2.filter2D(gray, -1, sharp_k)
flat = np.full((10, 10), 128, np.uint8)
flat_out = cv2.filter2D(flat, -1, lapl_k)
lap_f = cv2.Laplacian(f32, cv2.CV_32F)
blurred = cv2.GaussianBlur(gray, (0, 0), 3)
usm = cv2.addWeighted(gray, 1.6, blurred, -0.6, 0)
s_sharp, s_lapl = float(sharp_k.sum()), float(lapl_k.sum())
flat_max = int(np.abs(flat_out.astype(int)).max())
sharp_std = round(float(sharped.std()), 4)
sharp_mean = round(float(sharped.mean()), 4)
usm_w = round(1.6 + (-0.6), 6)
usm_std = round(float(usm.std()), 4)
print("锐化核 sum:", s_sharp, "| 锐化后 std:", sharp_std, "（原", round(float(gray.std()), 4), "）",
      "| 均值:", sharp_mean, "（原", round(float(gray.mean()), 4), "）")
print("拉普拉斯核 sum:", s_lapl, "| 纯常量图经它的输出绝对最大值:", flat_max)
print("Laplacian(CV_32F) 均值:", round(float(lap_f.mean()), 6),
      "min/max:", round(float(lap_f.min()), 2), round(float(lap_f.max()), 2))
print("unsharp 权重和:", usm_w, "| 后 std:", usm_std, "（原", round(float(gray.std()), 4), "）")
print("解读: 1) 锐化核 sum=1，所以亮度守恒（均值 116.1954 → 116.5697 只差 0.37），"
      "只是把 std 从 45.59 拉到 60.71；2) 拉普拉斯核 sum=0，常量区域输出恒 0——"
      "这是「边缘检测」的数学本质：对平坦区域不响应；3) unsharp mask 的权重和也是 1，"
      "（1.6 − 0.6 = 1）所以亮度不漂，只是把高频放大")

assert s_sharp == 1.0 and s_lapl == 0.0 and flat_max == 0
assert sharp_std == 60.7078 and sharp_mean == 116.5697
assert usm_w == 1.0 and usm_std == 50.7266

## 任务 7：输出类型

uint8 截断 vs float32 保负。

In [ ]:
edge_k = np.array([[1, 0, -1]], np.float32)
dx_u8 = cv2.filter2D(gray, -1, edge_k)
dx_f = cv2.filter2D(f32, -1, edge_k)
u8_min, u8_max = int(dx_u8.min()), int(dx_u8.max())
f_min, f_max = round(float(dx_f.min()), 2), round(float(dx_f.max()), 2)
n_neg = int((dx_f < 0).sum())
n_zero = int((dx_u8 == 0).sum())
abs_mean_f = round(float(np.abs(dx_f).mean()), 4)
mean_u8 = round(float(dx_u8.mean()), 4)
print("dx 核 {1,0,-1}: uint8 输出", u8_min, "~", u8_max, "| float32 输出", f_min, "~", f_max)
print("float 里负值像素:", n_neg, "| uint8 输出里等于 0 的:", n_zero)
print("abs(float).mean():", abs_mean_f, "vs uint8.mean():", mean_u8)
print("解读: 同一组卷积，uint8 把 −202 直接截成 0（负梯度全丢），输出均值只剩 uint8 的 5.4454；"
      "float32 的 |响应| 均值是 10.971，才是真实的梯度总量。"
      "所以方向性滤波（Sobel/Laplacian/自定义核）一律先转 float32，"
      "或者用 cv2.filter2D(gray, cv2.CV_32F, k) 让输出保持浮点")

assert u8_min == 0 and u8_max == 196 and f_min == -202.0 and f_max == 196.0
assert n_neg == 70003 and n_zero == 123276
assert abs_mean_f == 10.971 and mean_u8 == 5.4454

## 任务 8：核大小与平滑强度

循环骨架已给好，块内写 2 行。

In [ ]:
sizes = [3, 7, 15, 31]
blur_std = {}
blur_gap = {}
for ks in sizes:
    b = cv2.blur(f32, (ks, ks))
    blur_std[ks] = round(float(b.std()), 4)
for ks in sizes:
    print(f"   blur({ks:2d},{ks:2d}) std: {blur_std[ks]}")
d = round(blur_std[3] - blur_std[31], 4)
print("   std 从 3x3 到 31x31 降了", d, "| 核面积涨了", (31 * 31) / (3 * 3), "倍")
print("解读: 核大小与平滑强度不是线性关系——面积涨 106.8 倍，std 只降 7.37。"
      "因为均值滤波的方差按 1/核面积 递减，收益是开根号级的，"
      "所以「再糊一点」的边际代价很高、边际收益很低。选核大小要跟目标尺度匹配："
      "小目标用 3x3，去大块噪声才考虑 15 以上")

assert blur_std == {3: 43.4368, 7: 41.4624, 15: 39.0128, 31: 36.0681}
assert d == 7.3687

## 自查清单

- [ ] 能说出 `filter2D` 做的是相关还是卷积，以及怎么得到真卷积
- [ ] 会通过 `k.sum()` 判断一张滤波结果会不会变亮/变暗
- [ ] 知道 `sigma=0` 时高斯核长什么样
- [ ] 能解释 `sepFilter2D` 为什么快
- [ ] 知道 `REFLECT` 与 `REPLICATE` 在半径 1 时等价的原因
- [ ] 会用「与真值的差」而不是 `std` 评估去噪
- [ ] 能说出 sum=0 的核为什么能做边缘检测
- [ ] 知道方向性滤波必须用 float32 的原因